In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import json

# Authenticate locally with: gcloud auth application-default login
# Uses your Google account permissions; do not create or paste a service-account key.
cred = credentials.ApplicationDefault()

# Initialize the Firebase Admin SDK for this project explicitly.
try:
    app = firebase_admin.get_app("moonshot-export")
except ValueError:
    app = firebase_admin.initialize_app(
        cred, {"projectId": "energy-moonshot-tracker"}, name="moonshot-export"
    )
if app.project_id != "energy-moonshot-tracker":
    raise RuntimeError("The moonshot-export Firebase app points to a different project.")

# Initialize Firestore
db = firestore.client(app=app)

# Function to export a Firestore collection
def export_collection(collection_name):
    # Reference to the collection
    collection_ref = db.collection(collection_name)
    
    # Get all documents in the collection
    docs = collection_ref.stream()

    # Dictionary to store the collection data
    collection_data = {}

    # Loop through each document and save its data
    for doc in docs:
        collection_data[doc.id] = doc.to_dict()

    # Save the collection data to a JSON file
    file_name = f'{collection_name}.json'
    with open(file_name, 'w') as json_file:
        json.dump(collection_data, json_file, indent=2)

    print(f'Collection {collection_name} has been exported to {file_name}')

# Call the function with the collection name
export_collection('your-collection-name')  # Replace with your Firestore collection name
